# Onboard Camera Check

This notebook does a basic health check of the JetBot's onboard CSI camera.

What it does:
- Opens the camera using the stock `jetbot` package when available.
- Falls back to `jetcam` only if needed.
- Captures and displays one still frame.
- Starts a live preview in the notebook.
- Includes a cleanup cell to release the camera cleanly.

If the camera is busy or the preview stays blank:
- Shut down other notebooks that may be using the camera.
- Restart this notebook kernel and run the cells again.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

camera = None
camera_backend = None


def stop_existing_camera():
    global camera
    try:
        camera.unobserve_all()
    except Exception:
        pass
    try:
        camera.running = False
    except Exception:
        pass
    try:
        camera.stop()
    except Exception:
        pass


stop_existing_camera()

try:
    from jetbot import Camera, bgr8_to_jpeg

    camera = Camera.instance(width=224, height=224)
    camera_backend = "jetbot.Camera"
except ImportError:
    from jetcam.csi_camera import CSICamera
    from jetcam.utils import bgr8_to_jpeg

    camera = CSICamera(
        width=224,
        height=224,
        capture_width=1280,
        capture_height=720,
        capture_fps=30,
    )
    camera_backend = "jetcam.CSICamera"

print(f"Camera initialized using {camera_backend}.")


In [ ]:
frame = camera.value if getattr(camera, "value", None) is not None else camera.read()
print(f"Frame shape: {frame.shape}")
print(f"Frame dtype: {frame.dtype}")

still_image = widgets.Image(value=bgr8_to_jpeg(frame), format="jpeg", width=448, height=448)
display(still_image)


## Live Preview

Run the next cell to start a live camera preview. When you are done, run the cleanup cell near the end of the notebook.

In [ ]:
image_widget = widgets.Image(format="jpeg", width=448, height=448)
display(image_widget)


def update_image(change):
    image_widget.value = bgr8_to_jpeg(change["new"])


camera.unobserve_all()
camera.observe(update_image, names="value")
if hasattr(camera, "running"):
    camera.running = True
image_widget.value = bgr8_to_jpeg(camera.value)
print("Live preview started.")


## Optional Snapshot Save

Run this if you want to save a test image into the repo's `images/` folder.

In [ ]:
from pathlib import Path
import cv2
from datetime import datetime


snapshot = camera.value if getattr(camera, "value", None) is not None else camera.read()
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = Path("../images") / f"camera_check_{timestamp}.jpg"
cv2.imwrite(str(output_path), snapshot)
print(f"Saved snapshot to {output_path}")


## Cleanup

Run this after testing so the camera is released for other notebooks.

In [ ]:
camera.unobserve_all()
if hasattr(camera, "running"):
    camera.running = False
if hasattr(camera, "stop"):
    camera.stop()
print("Camera preview stopped.")
